# VRC-3PO revision — R1e: matched comparison with the published detectors

Two additions to R1d, written into the **same run store** (`vrc3po_revision/litevr`):

1. **Our CNN ensemble under their protocol** (10-fold over windows on Simulations) and under participant CV on Simulations → the table compares ours and theirs under both protocols on identical windows.
2. **FMS regression** with the LiteVR LSTM (`litevr_reg:LSTM`, LiteVR also regressed FMS) and our CNN (`reg:CNN`) under both protocols → their reported R² 0.92 against our negative held-out R², same models, only the split changes.

Runtime on a T4 ≈ 60–80 min (Step 4 optional, +40 min). Resumable. Every cell is self-contained, so a Colab restart only needs Step 0 re-run.

In [ ]:
# ═══ 0 · Setup (re-run after any restart) ═══
!pip install -q keras-tcn interpret-core pytest
from google.colab import drive, files
from pathlib import Path
import zipfile, shutil, os, glob
import pandas as pd, numpy as np, tensorflow as tf
drive.mount('/content/drive')
hits = glob.glob('/content/drive/MyDrive/**/vrc3po_benchmark_v1_1.csv', recursive=True)
assert hits, 'Benchmark not found: sign this Colab in to the Google account used for the earlier runs.'
REV = Path(hits[0]).parent; V11 = REV / 'vrc3po_benchmark_v1_1.csv'; OUT = REV / 'litevr'
assert (OUT / 'runs').exists(), 'R1d run store not found'
BUNDLE = REV / 'vrc3po_revision_bundle_v5.zip'
if not BUNDLE.exists():
    local = sorted(glob.glob('/content/vrc3po_revision_bundle_v5*.zip'))
    if not local:
        print('Upload vrc3po_revision_bundle_v5.zip from the VRC-3PO folder on your Mac:')
        local = [n for n in files.upload() if n.endswith('.zip')]
    shutil.copy(local[-1], BUNDLE)
root = Path('/content/vrc3po_code'); shutil.rmtree(root, ignore_errors=True); root.mkdir()
zipfile.ZipFile(BUNDLE).extractall(root); os.chdir(root)
assert 'litevr_reg' in (root / 'analysis/revision_cv.py').read_text(), 'wrong bundle: needs v5'
print('REV =', REV, '| runs in store:', len(list((OUT / 'runs').glob('*.npz'))),
      '| GPU:', tf.config.list_physical_devices('GPU'))
!python -m pytest tests -q -x

## Step 1 · Smoke test (~2 min)

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$REV/smoke_r1e" --scope simulations --protocol windows --models cnn_ensemble litevr_reg:LSTM reg:CNN --members 42 --repeats 1 --folds 2 --epochs 2
!python -m analysis.litevr regression --dataset "$V11" --output-dir "$REV/smoke_r1e" --bootstrap 20
shutil.rmtree(REV / 'smoke_r1e', ignore_errors=True); print('Smoke test passed.')

## Step 2 · Our CNN ensemble on Simulations under both protocols · ≈ 30–40 min

In [ ]:
assert tf.config.list_physical_devices('GPU'), 'Switch to a GPU runtime.'
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope simulations --protocol windows --folds 10 --repeats 1 --models cnn_ensemble
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope simulations --protocol cv --folds 5 --repeats 3 --models cnn_ensemble

## Step 3 · FMS regression on Simulations under both protocols (LiteVR LSTM and our CNN) · ≈ 30–40 min

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope simulations --protocol windows --folds 10 --repeats 1 --models litevr_reg:LSTM reg:CNN
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope simulations --protocol cv --folds 5 --repeats 3 --models litevr_reg:LSTM reg:CNN

## Step 4 · (optional) Regression on the 47 single-task participants, participant CV · ≈ 40 min

In [ ]:
RUN_STEP4 = False
if RUN_STEP4:
    !python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope seated --protocol cv --folds 5 --repeats 5 --models litevr_reg:LSTM reg:CNN

## Step 5 · Summaries → `results_revision_r1e.zip` (Drive + download) and printed tables

In [ ]:
import subprocess
for cmd in (['python', '-m', 'analysis.revision_cv', 'summarize', '--dataset', str(V11), '--output-dir', str(OUT), '--bootstrap', '2000'],
            ['python', '-m', 'analysis.litevr', 'regression', '--dataset', str(V11), '--output-dir', str(OUT), '--bootstrap', '2000']):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if line.startswith('[') or 'r2' in line or 'reg' in line: print(line, end='')
    assert proc.wait() == 0, 'summary failed: ' + ' '.join(cmd[2:4])
S = OUT / 'summary'
p = pd.read_csv(S / 'primary_estimates.csv'); p = p[(p.label == 'fms_gt2') & (p.scope == 'simulations')]
cols = ['protocol', 'model', 'repeats', 'auc_mean', 'auc_ci_low', 'auc_ci_high',
        'within_condition_auc', 'within_condition_ci_low', 'within_condition_ci_high']
print('\n=== SIMULATIONS_AUC ==='); print(p[cols].sort_values(['protocol', 'model']).round(3).to_csv(index=False))
print('=== REGRESSION ==='); print(pd.read_csv(S / 'regression.csv').round(3).to_csv(index=False))
zip_path = shutil.make_archive(str(REV / 'results_revision_r1e'), 'zip', S)
print('saved to Drive:', zip_path)
files.download(shutil.copy(zip_path, '/content/results_revision_r1e.zip'))

**Send back:** the downloaded `results_revision_r1e.zip`, or paste the two printed blocks (`SIMULATIONS_AUC`, `REGRESSION`).